<a href="https://colab.research.google.com/github/bumsootead/Seoul_Subway_Analytics/blob/main/Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
RAW_2024 = r"/content/서울교통공사_역별 일별 시간대별 승하차인원 정보_20241231.csv"


RAW_2025 = r"/content/서울교통공사_1_8호선 역별 일별 시간대별 승객유형별 승하차인원_20251231.csv"


In [ ]:
pip install pandas numpy pyarrow sqlalchemy psycopg2-binary scikit-learn seaborn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 22.6 MB/s eta 0:00:00


# 2024 file: already contains total passenger counts by date, line, station, direction, and hour.

# 2025 file: contains counts split by passenger type



In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW_2024 = Path(
    r"/content/서울교통공사_역별 일별 시간대별 승하차인원 정보_20241231.csv"
)

RAW_2025 = Path(
    r"/content/서울교통공사_1_8호선 역별 일별 시간대별 승객유형별 승하차인원_20251231.csv"
)

OUTPUT_DIR = Path("data/cleaned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df_2024_raw = pd.read_csv(RAW_2024, encoding="cp949")
df_2025_raw = pd.read_csv(RAW_2025, encoding="cp949")

print(df_2024_raw.shape)
print(df_2025_raw.shape)

print(df_2024_raw.columns.tolist())
print(df_2025_raw.columns.tolist())

(199424, 26)
(797226, 27)
['연번', '수송일자', '호선', '역번호', '역명', '승하차구분', '06시이전', '06-07시간대', '07-08시간대', '08-09시간대', '09-10시간대', '10-11시간대', '11-12시간대', '12-13시간대', '13-14시간대', '14-15시간대', '15-16시간대', '16-17시간대', '17-18시간대', '18-19시간대', '19-20시간대', '20-21시간대', '21-22시간대', '22-23시간대', '23-24시간대', '24시이후']
['연번', '수송일자', '호선명', '역번호', '역명', '승하차구분', '승객유형', '06시간대이전', '06-07시간대', '07-08시간대', '08-09시간대', '09-10시간대', '10-11시간대', '11-12시간대', '12-13시간대', '13-14시간대', '14-15시간대', '15-16시간대', '16-17시간대', '17-18시간대', '18-19시간대', '19-20시간대', '20-21시간대', '21-22시간대', '22-23시간대', '23-24시간대', '24시간대이후']


Define common names for hourly fields

In [ ]:
DIRECTION_MAP = {
    "승차": "board",
    "하차": "alight"
}

KOREAN_HOUR_MAP = {
    # 2025 naming
    "06시간대이전": "before_06",
    "24시간대이후": "after_24",

    # 2024 naming
    "06시이전": "before_06",
    "24시이후": "after_24",

    # Shared naming
    "06-07시간대": "06_07",
    "07-08시간대": "07_08",
    "08-09시간대": "08_09",
    "09-10시간대": "09_10",
    "10-11시간대": "10_11",
    "11-12시간대": "11_12",
    "12-13시간대": "12_13",
    "13-14시간대": "13_14",
    "14-15시간대": "14_15",
    "15-16시간대": "15_16",
    "16-17시간대": "16_17",
    "17-18시간대": "17_18",
    "18-19시간대": "18_19",
    "19-20시간대": "19_20",
    "20-21시간대": "20_21",
    "21-22시간대": "21_22",
    "22-23시간대": "22_23",
    "23-24시간대": "23_24"
}

HOUR_BUCKETS = [
    "before_06", "06_07", "07_08", "08_09", "09_10",
    "10_11", "11_12", "12_13", "13_14", "14_15",
    "15_16", "16_17", "17_18", "18_19", "19_20",
    "20_21", "21_22", "22_23", "23_24", "after_24"
]

In [ ]:
def clean_to_hourly_long(df, source_year, has_passenger_type):
    df = df.copy()

    # Standardize identifier names.
    df = df.rename(columns={
        "수송일자": "service_date",
        "호선명": "line_name",
        "호선": "line_name",
        "역번호": "station_code",
        "역명": "station_name",
        "승하차구분": "direction",
        "승객유형": "passenger_type"
    })

    # Supplier row number is not analytically useful.
    df = df.drop(columns=["연번"], errors="ignore")

    # Clean dimensions.
    df["service_date"] = pd.to_datetime(df["service_date"], errors="coerce")
    df["line_name"] = df["line_name"].astype(str).str.strip()
    df["station_code"] = df["station_code"].astype(str).str.strip()
    df["station_name"] = df["station_name"].astype(str).str.strip()
    df["direction"] = df["direction"].map(DIRECTION_MAP)

    # 2024 has no passenger-type detail.
    if has_passenger_type:
        df["passenger_type"] = df["passenger_type"].astype(str).str.strip()
    else:
        df["passenger_type"] = "all_passengers"

    # Identify and clean hour columns.
    hour_columns = [column for column in df.columns if column in KOREAN_HOUR_MAP]

    for column in hour_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce").fillna(0)
        df[column] = df[column].astype("int64")

    # Convert 20 separate hour columns into rows.
    hourly_long = df.melt(
        id_vars=[
            "service_date",
            "line_name",
            "station_code",
            "station_name",
            "direction",
            "passenger_type"
        ],
        value_vars=hour_columns,
        var_name="raw_hour_bucket",
        value_name="passenger_count"
    )

    hourly_long["hour_bucket"] = hourly_long["raw_hour_bucket"].map(
        KOREAN_HOUR_MAP
    )

    # Add calendar fields for analysis and Tableau.
    hourly_long["source_year"] = source_year
    hourly_long["day_of_week"] = hourly_long["service_date"].dt.day_name()
    hourly_long["is_weekday"] = hourly_long["service_date"].dt.dayofweek < 5
    hourly_long["is_weekend"] = ~hourly_long["is_weekday"]
    hourly_long["month"] = hourly_long["service_date"].dt.month
    hourly_long["year_month"] = (
        hourly_long["service_date"].dt.to_period("M").astype(str)
    )

    return hourly_long[
        [
            "service_date",
            "source_year",
            "line_name",
            "station_code",
            "station_name",
            "direction",
            "passenger_type",
            "hour_bucket",
            "passenger_count",
            "day_of_week",
            "is_weekday",
            "is_weekend",
            "month",
            "year_month"
        ]
    ]

# Clean each dataset separately

In [ ]:
station_hourly_2024 = clean_to_hourly_long(
    df_2024_raw,
    source_year=2024,
    has_passenger_type=False
)

station_hourly_2025_by_type = clean_to_hourly_long(
    df_2025_raw,
    source_year=2025,
    has_passenger_type=True
)


# Create the common standardized aggregate layer
Aggregate every passenger type in 2025. This puts 2025 at the same level as 2024.

In [ ]:
station_hourly_2025_total = (
    station_hourly_2025_by_type
    .groupby(
        [
            "service_date", "source_year",
            "line_name", "station_code", "station_name",
            "direction", "hour_bucket",
            "day_of_week", "is_weekday", "is_weekend",
            "month", "year_month"
        ],
        as_index=False
    )["passenger_count"]
    .sum()
)

station_hourly_2025_total["passenger_type"] = "all_passengers"

station_hourly_2025_total = station_hourly_2025_total[
    station_hourly_2024.columns
]



# Validate 2025 aggregation


In [ ]:
typed_total = (
    station_hourly_2025_by_type
    .groupby(
        [
            "service_date", "line_name", "station_code",
            "direction", "hour_bucket"
        ],
        as_index=False
    )["passenger_count"]
    .sum()
    .rename(columns={"passenger_count": "from_passenger_types"})
)

aggregate_total = (
    station_hourly_2025_total
    .groupby(
        [
            "service_date", "line_name", "station_code",
            "direction", "hour_bucket"
        ],
        as_index=False
    )["passenger_count"]
    .sum()
    .rename(columns={"passenger_count": "from_aggregate_layer"})
)

validation = typed_total.merge(
    aggregate_total,
    on=[
        "service_date", "line_name", "station_code",
        "direction", "hour_bucket"
    ]
)

assert (
    validation["from_passenger_types"]
    == validation["from_aggregate_layer"]
).all()

# Append the standardized 2024 and 2025 layers

In [ ]:
station_hourly_total = pd.concat(
    [station_hourly_2024, station_hourly_2025_total],
    ignore_index=True
).sort_values(
    [
        "service_date", "line_name", "station_code",
        "direction", "hour_bucket"
    ]
).reset_index(drop=True)

print(station_hourly_total.shape)
print(station_hourly_total.head())
print(station_hourly_total.tail())


(5997760, 14)
  service_date  source_year line_name station_code station_name direction  \
0   2024-01-01         2024       1호선          150          서울역    alight   
1   2024-01-01         2024       1호선          150          서울역    alight   
2   2024-01-01         2024       1호선          150          서울역    alight   
3   2024-01-01         2024       1호선          150          서울역    alight   
4   2024-01-01         2024       1호선          150          서울역    alight   

   passenger_type hour_bucket  passenger_count day_of_week  is_weekday  \
0  all_passengers       06_07              867      Monday        True   
1  all_passengers       07_08              834      Monday        True   
2  all_passengers       08_09             1201      Monday        True   
3  all_passengers       09_10             1744      Monday        True   
4  all_passengers       10_11             1731      Monday        True   

   is_weekend  month year_month  
0       False      1    2024-01  
1       Fa

In [ ]:
station_hourly_total.to_parquet(
    OUTPUT_DIR / "station_hourly_total.parquet",
    index=False
)

station_hourly_total.to_csv(
    OUTPUT_DIR / "station_hourly_total.csv",
    index=False,
    encoding="utf-8-sig"
)

station_hourly_2025_by_type.to_parquet(
    OUTPUT_DIR / "station_hourly_passenger_type_2025.parquet",
    index=False
)

# Create a daily station output

In [ ]:
station_daily = (
    station_hourly_total
    .groupby(
        [
            "service_date", "source_year",
            "line_name", "station_code", "station_name",
            "direction", "day_of_week",
            "is_weekday", "is_weekend",
            "month", "year_month"
        ],
        as_index=False
    )["passenger_count"]
    .sum()
)

station_daily_wide = (
    station_daily
    .pivot_table(
        index=[
            "service_date", "source_year",
            "line_name", "station_code", "station_name",
            "day_of_week", "is_weekday", "is_weekend",
            "month", "year_month"
        ],
        columns="direction",
        values="passenger_count",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
    .rename(columns={
        "board": "boardings",
        "alight": "alightings"
    })
)

station_daily_wide["total_activity"] = (
    station_daily_wide["boardings"]
    + station_daily_wide["alightings"]
)

station_daily_wide["net_boarding"] = (
    station_daily_wide["boardings"]
    - station_daily_wide["alightings"]
)

station_daily_wide.to_csv(
    OUTPUT_DIR / "station_daily_wide.csv",
    index=False,
    encoding="utf-8-sig"
)


# Overall daily ridership trend

In [ ]:
daily_citywide = (
    station_daily
    .groupby(["service_date", "source_year"], as_index=False)["passenger_count"]
    .sum()
)

daily_citywide.head()

,service_date,source_year,passenger_count
0,2024-01-01,2024,3804860
1,2024-01-02,2024,9029451
2,2024-01-03,2024,9381398
3,2024-01-04,2024,9561173
4,2024-01-05,2024,9856618


# Top stations by total ridership

In [ ]:
top_stations = (
    station_daily_wide
    .groupby(["source_year", "line_name", "station_name"], as_index=False)
    ["total_activity"]
    .sum()
    .sort_values("total_activity", ascending=False)
)

print(top_stations.head(20))

     source_year line_name station_name  total_activity
52          2024       2호선     잠실(송파구청)        58859834
10          2024       2호선           강남        56462439
59          2024       2호선         홍대입구        56244605
14          2024       2호선      구로디지털단지        40224826
2           2024       1호선          서울역        39978774
37          2024       2호선           신림        39507874
26          2024       2호선     삼성(무역센터)        38830203
36          2024       2호선          신도림        37048193
63          2024       3호선        고속터미널        36954059
30          2024       2호선           선릉        36514563
43          2024       2호선           역삼        36394175
50          2024       2호선        을지로입구        35449760
28          2024       2호선  서울대입구(관악구청)        33241760
31          2024       2호선           성수        33208930
25          2024       2호선           사당        31723497
326         2025       2호선     잠실(송파구청)        29374223
212         2024       7호선      가산디지털단지        2

# Weekday versus weekend activity

In [ ]:
weekday_weekend = (
    station_daily_wide
    .groupby(["station_name", "line_name", "is_weekday"], as_index=False)
    ["total_activity"]
    .mean()
    .rename(columns={"total_activity": "average_daily_activity"})
)

print(weekday_weekend.head())

  station_name line_name  is_weekday  average_daily_activity
0         가락시장       3호선       False            11774.666667
1         가락시장       3호선        True            20057.489848
2         가락시장       8호선       False            11275.538462
3         가락시장       8호선        True            19438.312183
4      가산디지털단지       7호선       False            25077.352564


# 2025 covers only July–December. Therefore compare it with July–December 2024, not all of 2024.

In [ ]:
comparable_period = station_daily_wide[
    station_daily_wide["month"].between(7, 12)
].copy()

yoy_comparison = (
    comparable_period
    .groupby(
        ["source_year", "line_name", "station_code", "station_name"],
        as_index=False
    )["total_activity"]
    .sum()
    .pivot_table(
        index=["line_name", "station_code", "station_name"],
        columns="source_year",
        values="total_activity"
    )
    .reset_index()
)

yoy_comparison["yoy_pct_change"] = (
    (yoy_comparison[2025] - yoy_comparison[2024])
    / yoy_comparison[2024]
) * 100

In [ ]:
passenger_type_summary = (
    station_hourly_2025_by_type
    .groupby("passenger_type", as_index=False)["passenger_count"]
    .sum()
    .sort_values("passenger_count", ascending=False)
)

passenger_type_summary["share_pct"] = (
    passenger_type_summary["passenger_count"]
    / passenger_type_summary["passenger_count"].sum()
    * 100
)

print(passenger_type_summary)

   passenger_type  passenger_count  share_pct
4              일반       1296816940  78.673625
3             우대권        287103976  17.417655
11            청소년         42601698   2.584505
10             직원          9117688   0.553140
0             어린이          8465991   0.513604
2           영어 일반          1654433   0.100369
9          중국어 일반          1282371   0.077797
6           일어 일반           941971   0.057146
1          영어 어린이           140879   0.008547
8         중국어 어린이           108557   0.006586
7             중고생            60027   0.003642
5          일어 어린이            55771   0.003383
